# **Customer Churn Prediction** 


## **Problem Statement:**

Develop a predictive model to identify customers at risk of churning from an investment bank, enabling proactive retention strategies to minimize customer loss and maximize revenue growth.


## **About the Dataset**

There are 14 columns/features and 10k rows/samples.

**RowNumber**—corresponds to the record (row) number and has no effect on the output.

**CustomerId**—contains random values and has no effect on customer leaving the bank.

**Surname**—the surname of a customer has no impact on their decision to leave the bank.

**CreditScore**—can have an effect on customer churn, since a customer with a higher credit score is less likely to leave the bank.

**Geography**—a customer’s location can affect their decision to leave the bank.

**Gender**—it’s interesting to explore whether gender plays a role in a customer leaving the bank.

**Age**—this is certainly relevant, since older customers are less likely to leave their bank than younger ones.

**Tenure**—refers to the number of years that the customer has been a client of the bank. Normally, older clients are more loyal and less likely to leave a bank.

**Balance**—also a very good indicator of customer churn, as people with a higher balance in their accounts are less likely to leave the bank compared to those with lower balances.

**NumOfProducts**—refers to the number of products that a customer has purchased through the bank.

**HasCrCard**—denotes whether or not a customer has a credit card. This column is also relevant, since people with a credit card are less likely to leave the bank.

**IsActiveMember**—active customers are less likely to leave the bank.

**EstimatedSalary**—as with balance, people with lower salaries are more likely to leave the bank compared to those with higher salaries.

**Exited**—whether or not the customer left the bank.


## **KNN**

The K-Nearest Neighbors (KNN) algorithm is a simple and effective machine learning technique that classifies data points by finding the K most similar instances to a new input and voting for the target class or value.

### **The most commonly used hyperparameters for K-Nearest Neighbors (KNN) algorithm:**

n_neighbors: The number of nearest neighbors to consider when making a prediction. Increasing this number can improve the model's performance, but also increases the computation time.

weights: The weight function used to calculate the distance between samples. Supported weights are 'uniform' (all points have equal weight) and 'distance' (points closer to the query point have higher weight).

algorithm: The algorithm used to compute the nearest neighbors. Supported algorithms are 'brute' (exhaustive search), 'kd_tree' (k-d tree search), and 'ball_tree' (ball tree search).

leaf_size: The number of samples in each leaf node of the k-d tree or ball tree. Increasing this number can improve the model's performance, but also increases the computation time.

p: The power parameter for the Minkowski metric. When p=1, it is the Manhattan distance, and when p=2, it is the Euclidean distance.

metric: The distance metric used to calculate the distance between samples. Supported metrics are 'minkowski' (Minkowski distance), 'euclidean' (Euclidean distance), 'manhattan' (Manhattan distance), and 'chebyshev' (Chebyshev distance).

### **Here are some common values for these hyperparameters:**

n_neighbors: 3, 5, 10, 20

weights: 'uniform', 'distance'

algorithm: 'brute', 'kd_tree', 'ball_tree'

leaf_size: 10, 20, 30

p: 1, 2

metric: 'minkowski', 'euclidean', 'manhattan', 'chebyshev'


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.svm import SVC
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
data = pd.read_excel('Churn.xlsx')

In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [5]:
# is null?
isnull = data.isnull().sum()
isnull

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [6]:
# Preprocess data
selected_features = [
    'CreditScore', 'Geography', 'Gender', 'Age',
    'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', #independent Variables
    'IsActiveMember', 'EstimatedSalary'
]
X = data[selected_features]
y = data[['Exited']]# dependent Variable

### **Encoding of Categorical Variables:**
Categorical variables are variables that represent categories or groups and do not have a numerical value. Machine learning algorithms require numerical inputs, so categorical variables need to be encoded into a numerical format. Common encoding techniques include:

- **One-Hot Encoding**: One-hot encoding converts categorical variables into binary vectors where each category is represented by a binary variable (0 or 1). It creates a new binary feature for each category, with only one feature having a value of 1 (hot) and the others having a value of 0 (cold).

- **Label Encoding**: Label encoding assigns a unique integer value to each category in the categorical variable. It is suitable for ordinal categorical variables where the categories have a meaningful order. However, it may not be suitable for nominal categorical variables as it implies ordinality. No order . 

- **Ordinal Encoding**: Ordinal encoding is similar to label encoding but preserves the order of categories by mapping them to ordered integers. It is suitable for ordinal categorical variables with a clear order. Has Order.

In [7]:
# Label encoding
le = LabelEncoder()
X['Geography'] = le.fit_transform(X['Geography'])
X['Gender'] = le.fit_transform(X['Gender'])

### **Scaling or Normalization:**
Scaling or normalization is the process of standardizing the range of features or variables in the dataset. It ensures that all features contribute equally to the analysis and prevents features with larger scales from dominating the model. Common scaling techniques include:

- **Standardization**: Standardization (Z-score normalization) scales the features to have a mean of 0 and a standard deviation of 1. It subtracts the mean and divides by the standard deviation of each feature.

- **Min-Max Scaling**: Min-Max scaling scales the features to a fixed range, typically between 0 and 1. It subtracts the minimum value and divides by the range (maximum value minus minimum value) of each feature.

- **Robust Scaling**: Robust scaling scales the features using the median and interquartile range (IQR) instead of the mean and standard deviation. It is more robust to outliers compared to standardization.

In [8]:
# Scaling
scaler = MinMaxScaler()
X[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']] = scaler.fit_transform(X[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']])


In [9]:
# Split data
train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=0, train_size=0.8
)

In [10]:
# Train model
model = KNeighborsClassifier(n_neighbors=2, metric='euclidean', weights='uniform', algorithm='auto', leaf_size=50, p=2)
model.fit(train_X, train_y)

KNeighborsClassifier(leaf_size=50, metric='euclidean', n_neighbors=2)

In [11]:
# Evaluate model
val_prediction = model.predict(val_X)
y_pred_proba = model.predict_proba(val_X)[:,1]
accuracy = accuracy_score(val_y, val_prediction)
print(f'Model accuracy: {accuracy}')

Model accuracy: 0.817


In [33]:
print(confusion_matrix(val_y, val_prediction))
print(classification_report(val_y, val_prediction))

[[1578   17]
 [ 271  134]]
              precision    recall  f1-score   support

           0       0.85      0.99      0.92      1595
           1       0.89      0.33      0.48       405

    accuracy                           0.86      2000
   macro avg       0.87      0.66      0.70      2000
weighted avg       0.86      0.86      0.83      2000



In [13]:
auc = roc_auc_score(val_y, y_pred_proba)
print(auc)

0.7075134486628739


In [14]:
# Save model
joblib.dump(model, 'churn_model.pkl')

['churn_model.pkl']

### **OOP Approach**

In [25]:

class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None
        self.scaler = MinMaxScaler()

    # Load Excel
    def load_data(self):
        self.data = pd.read_excel(self.file_path)
        print("Data loaded!")

    #  Preprocess
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]

        #  1D target
        self.y = self.data['Exited']

        # Proper encoding
        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)

        print("Preprocessing done!")

    #  Split first
    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=0
        )
        print("Data split!")

    #  Scale AFTER split (important)
    def scale_data(self):
        self.train_X = self.scaler.fit_transform(self.train_X)
        self.val_X = self.scaler.transform(self.val_X)
        print("Scaling done!")

    #  Train KNN
    def train_model(self):
        self.model = KNeighborsClassifier(n_neighbors=5)
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    #  Evaluate
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)

        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")


# Usage

churn = ChurnPrediction('Churn.xlsx')  # correct

churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.scale_data()   #  important step for KNN
churn.train_model()
accuracy, auc = churn.evaluate_model()

churn.save_model('churn_model.pkl')

Data loaded!
Preprocessing done!
Data split!
Scaling done!
Model trained!
Model accuracy: 0.8145
Model AUC score: 0.7593
Model saved!


### **Procedural Approach**

In [26]:

#  Load Excel
def load_data(file_path):
    return pd.read_excel(file_path)


#  Preprocess (NO scaling yet)
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]
    y = data['Exited']   #  1D

    #  Proper encoding
    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


# Split first
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


# Scale AFTER split
def scale_data(train_X, val_X):
    scaler = MinMaxScaler()
    train_X = scaler.fit_transform(train_X)
    val_X = scaler.transform(val_X)
    return train_X, val_X


#  Train
def train_model(train_X, train_y):
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(train_X, train_y)
    return model


#  Evaluate (fixed AUC)
def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)

    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    #  Use probabilities for AUC
    y_pred_proba = model.predict_proba(val_X)[:, 1]
    auc = roc_auc_score(val_y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# Save
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


#  Usage

file_path = 'Churn.xlsx'   #  same folder

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

train_X, val_X = scale_data(train_X, val_X)   # correct order

model = train_model(train_X, train_y)

accuracy, auc = evaluate_model(model, val_X, val_y)

save_model(model, 'churn_model.pkl')

Model accuracy: 0.8145
Model AUC score: 0.7593
Model saved!


### A Decision Tree Classifier
is a type of supervised learning algorithm in machine learning. It works by creating a tree-like model of decisions and their possible consequences, including chance event outcomes, resource costs, and utility. The tree is constructed by recursively partitioning the data into subsets based on the values of the input features.

### **The most commonly used hyperparameters for Decision Tree Classifier**:

criterion: The function to measure the quality of a split. Supported criteria are "gini" for the Gini impurity and "entropy" for the information gain.

max_depth: The maximum depth of the tree. Increasing this number can improve the model's performance, but also increases the risk of overfitting.

min_samples_split: The minimum number of samples required to split an internal node. Decreasing this number can lead to overfitting, while increasing it can lead to underfitting.

min_samples_leaf: The minimum number of samples required to be at a leaf node. Decreasing this number can lead to overfitting, while increasing it can lead to underfitting.

max_features: The maximum number of features to consider at each split. Increasing this number can improve the model's performance, but also increases the computation time.

random_state: The random seed used to shuffle the data before splitting it into training and testing sets. Setting this to a fixed value ensures reproducibility of the results.

class_weight: The weight assigned to each class during training. This can be useful for imbalanced datasets, where one class has a much larger number of instances than the others.

### **Here are some common values for these hyperparameters:**

criterion: 'gini', 'entropy'

max_depth: 3, 5, 10, None (None means no limit)

min_samples_split: 2, 5, 10

min_samples_leaf: 1, 5, 10

max_features: 'auto', 'sqrt', 'log2', None (None means no limit)

random_state: 0, 42, 100

class_weight: 'balanced', 'balanced_subsample', None (None means all classes are equal)


### **OOP Approach**

In [21]:


class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    #  Load Excel file
    def load_data(self):
        self.data = pd.read_excel(self.file_path)
        print("Data loaded successfully!")

    #  Preprocess
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]

        # Make y 1D (important fix)
        self.y = self.data['Exited']

        #  Encode categorical variables
        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)

        print("Preprocessing complete!")

    #  Split data
    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=0
        )
        print("Data split complete!")

    #  Train model
    def train_model(self):
        self.model = DecisionTreeClassifier(random_state=0)
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    #  Evaluate model
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)

        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    #  Save model
    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")

    #  Load model
    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print("Model loaded!")

# Usage
churn = ChurnPrediction('Churn.xlsx')  # file in same folder
churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.train_model()
accuracy, auc = churn.evaluate_model()

# Save model
churn.save_model('churn_model.pkl')

Data loaded successfully!
Preprocessing complete!
Data split complete!
Model trained!
Model accuracy: 0.8055
Model AUC score: 0.7224
Model saved!


### **Procedural Approach**

In [22]:

#  Load Excel file
def load_data(file_path):
    return pd.read_excel(file_path)


#  Preprocess
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]

    #  Make y 1D
    y = data['Exited']

    #  Encode categorical variables
    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


#  Split
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


#  Train
def train_model(X, y):
    model = DecisionTreeClassifier(random_state=0)
    model.fit(X, y)
    return model


#  Evaluate
def evaluate_model(model, X, y):
    val_prediction = model.predict(X)

    accuracy = accuracy_score(y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# Save
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


#  Usage

file_path = 'Churn.xlsx'   #  same folder

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model = train_model(train_X, train_y)

accuracy, auc = evaluate_model(model, val_X, val_y)

save_model(model, 'churn_model.pkl')

Model accuracy: 0.8055
Model AUC score: 0.7224
Model saved!


### Random Forest 
is a supervised learning algorithm that combines multiple decision trees to produce a more accurate and stable prediction model. It works by creating a collection of decision trees, where each tree is trained on a random subset of the training data. The final prediction is made by combining the predictions of all the trees.

### **The most commonly used hyperparameters for Random Forest Classifier:**

n_estimators: The number of trees in the forest. Increasing this number can improve the model's performance, but also increases the computation time.

criterion: The function to measure the quality of a split. Supported criteria are "gini" for the Gini impurity and "entropy" for the information gain.

max_depth: The maximum depth of each tree. Increasing this number can improve the model's performance, but also increases the risk of overfitting.

min_samples_split: The minimum number of samples required to split an internal node. Decreasing this number can lead to overfitting, while increasing it can lead to underfitting.

min_samples_leaf: The minimum number of samples required to be at a leaf node. Decreasing this number can lead to overfitting, while increasing it can lead to underfitting.

max_features: The maximum number of features to consider at each split. Increasing this number can improve the model's performance, but also increases the computation time.

max_leaf_nodes: The maximum number of leaf nodes in each tree. Increasing this number can improve the model's performance, but also increases the computation time.

min_impurity_decrease: The minimum decrease in impurity required to split an internal node. Increasing this number can lead to underfitting, while decreasing it can lead to overfitting.

bootstrap: Whether to use bootstrap sampling to build each tree. If True, each tree is built on a random subset of the training data.

oob_score: Whether to use out-of-bag samples to estimate the generalization accuracy.

random_state: The random seed used to shuffle the data before building each tree. Setting this to a fixed value ensures reproducibility of the results.

class_weight: The weight assigned to each class during training. This can be useful for imbalanced datasets, where one class has a much larger number of instances than the others.

### **Here are some common values for these hyperparameters:**

n_estimators: 10, 50, 100, 200

criterion: 'gini', 'entropy'

max_depth: 3, 5, 10, None (None means no limit)

min_samples_split: 2, 5, 10

min_samples_leaf: 1, 5, 10

max_features: 'auto', 'sqrt', 'log2', None (None means no limit)

max_leaf_nodes: 10, 50, 100, None (None means no limit)

min_impurity_decrease: 0.0, 0.1, 0.5

bootstrap: True, False

oob_score: True, False

random_state: 0, 42, 100

class_weight: 'balanced', 'balanced_subsample', None (None means all classes are equal)


### **OOP Approach**

In [27]:
class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None
        self.scaler = MinMaxScaler()

    # Load Excel
    def load_data(self):
        self.data = pd.read_excel(self.file_path)
        print("Data loaded!")

    #  Preprocess
    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]

        self.X = self.data[selected_features]

        #  1D target
        self.y = self.data['Exited']

        # Proper encoding
        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)

        print("Preprocessing done!")

    #  Split first
    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=0
        )
        print("Data split!")

    #  Scale AFTER split (important)
    def scale_data(self):
        self.train_X = self.scaler.fit_transform(self.train_X)
        self.val_X = self.scaler.transform(self.val_X)
        print("Scaling done!")

    #  Train KNN
    def train_model(self):
        self.model =  RandomForestClassifier(random_state=0)
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    #  Evaluate
    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)

        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")


# Usage

churn = ChurnPrediction('Churn.xlsx')  # correct

churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.scale_data()   #  important step for KNN
churn.train_model()
accuracy, auc = churn.evaluate_model()

churn.save_model('churn_model.pkl')

Data loaded!
Preprocessing done!
Data split!
Scaling done!
Model trained!
Model accuracy: 0.8670
Model AUC score: 0.8698
Model saved!


### **Procedural Approach**

In [28]:
#  Load Excel file
def load_data(file_path):
    return pd.read_excel(file_path)


#  Preprocess
def preprocess_data(data):
    selected_features = [
        'CreditScore', 'Geography', 'Gender', 'Age',
        'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
        'IsActiveMember', 'EstimatedSalary'
    ]

    X = data[selected_features]

    #  Make y 1D
    y = data['Exited']

    #  Encode categorical variables
    X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

    return X, y


#  Split
def split_data(X, y):
    return train_test_split(X, y, train_size=0.8, random_state=0)


#  Train
def train_model(X, y):
    model = RandomForestClassifier(random_state=0)
    model.fit(X, y)
    return model


#  Evaluate
def evaluate_model(model, X, y):
    val_prediction = model.predict(X)

    accuracy = accuracy_score(y, val_prediction)
    print(f'Model accuracy: {accuracy:.4f}')

    y_pred_proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_pred_proba)
    print(f'Model AUC score: {auc:.4f}')

    return accuracy, auc


# Save
def save_model(model, model_path):
    joblib.dump(model, model_path)
    print("Model saved!")


#  Usage

file_path = 'Churn.xlsx'   #  same folder

data = load_data(file_path)

X, y = preprocess_data(data)

train_X, val_X, train_y, val_y = split_data(X, y)

model = train_model(train_X, train_y)

accuracy, auc = evaluate_model(model, val_X, val_y)

save_model(model, 'churn_model.pkl')

Model accuracy: 0.8660
Model AUC score: 0.8699
Model saved!


### Support Vector Machine (SVM) 
is a supervised learning algorithm that can be used for classification and regression tasks. It works by finding the hyperplane that maximally separates the classes in the feature space.

### **The most commonly used hyperparameters for Support Vector Machines (SVMs) are:**

C: The regularization parameter. It controls the trade-off between the margin and the misclassification error.


kernel: The kernel function used to transform the data into a higher dimensional space.


gamma: The kernel coefficient. It is used to control the spread of the kernel.
degree: The degree of the polynomial kernel.


### **Here are some common values for these hyperparameters:**

C: 1.0, 10.0, 100.0, 1000.0

kernel: 'rbf', 'linear', 'poly', 'sigmoid'

gamma: 'scale', 'auto', 0.1, 1.0, 10.0

degree: 2, 3, 4, 5


### **OOP Approach**

In [31]:


class ChurnPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None
        self.scaler = MinMaxScaler()

    def load_data(self):
        self.data = pd.read_excel(self.file_path)  # Excel
        print("Data loaded!")

    def preprocess_data(self):
        selected_features = [
            'CreditScore', 'Geography', 'Gender', 'Age',
            'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
            'IsActiveMember', 'EstimatedSalary'
        ]
        self.X = self.data[selected_features]
        self.y = self.data['Exited']  # ✅ 1D

        self.X = pd.get_dummies(self.X, columns=["Geography", "Gender"], drop_first=True)
        print("Preprocessing done!")

    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, train_size=0.8, random_state=0
        )
        print("Data split!")

    def scale_data(self):
        self.train_X = self.scaler.fit_transform(self.train_X)
        self.val_X = self.scaler.transform(self.val_X)
        print("Scaling done!")

    def train_model(self):
        self.model = SVC(probability=True, random_state=0)
        self.model.fit(self.train_X, self.train_y)
        print("Model trained!")

    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy:.4f}')

        y_pred_proba = self.model.predict_proba(self.val_X)[:, 1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model AUC score: {auc:.4f}')

        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)
        print("Model saved!")

    def load_model(self, model_path):
        self.model = joblib.load(model_path)
        print("Model loaded!")

# =======================
# Usage
# =======================
churn = ChurnPrediction('Churn.xlsx')  # file in notebook folder
churn.load_data()
churn.preprocess_data()
churn.split_data()
churn.scale_data()      #  important for SVC
churn.train_model()
accuracy, auc = churn.evaluate_model()
churn.save_model('churn_model.pkl')

Data loaded!
Preprocessing done!
Data split!
Scaling done!
Model trained!
Model accuracy: 0.8560
Model AUC score: 0.8259
Model saved!


### **Procedural Approach**

In [32]:


# 1️ Load data

file_path = 'Churn.xlsx'   # file in notebook folder
data = pd.read_excel(file_path)
print("Data loaded!")


# 2️ Preprocess data

selected_features = [
    'CreditScore', 'Geography', 'Gender', 'Age',
    'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
    'IsActiveMember', 'EstimatedSalary'
]

X = data[selected_features]
y = data['Exited']   # 1D target

# One-hot encoding for categorical features
X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)
print("Preprocessing done!")


# 3️ Split data

train_X, val_X, train_y, val_y = train_test_split(
    X, y, train_size=0.8, random_state=0
)
print("Data split!")


# 4️ Scale data (important for SVC)

scaler = MinMaxScaler()
train_X = scaler.fit_transform(train_X)
val_X = scaler.transform(val_X)
print("Scaling done!")


# 5️ Train SVC

model = SVC(probability=True, random_state=0)
model.fit(train_X, train_y)
print("Model trained!")


# 6️Evaluate model

val_prediction = model.predict(val_X)
accuracy = accuracy_score(val_y, val_prediction)
print(f'Model accuracy: {accuracy:.4f}')

y_pred_proba = model.predict_proba(val_X)[:, 1]
auc = roc_auc_score(val_y, y_pred_proba)
print(f'Model AUC score: {auc:.4f}')


# 7️ Save model

joblib.dump(model, 'churn_svc_model.pkl')
print("Model saved!")

Data loaded!
Preprocessing done!
Data split!
Scaling done!
Model trained!
Model accuracy: 0.8560
Model AUC score: 0.8259
Model saved!
